# Experiment 6 – Containerization & API Deployment

## Aim
To package the final machine learning model in Docker and build a FastAPI service for predictions.

## Objectives
1. Create a FastAPI `/predict` endpoint.
2. Accept model features through a JSON POST request.
3. Return the predicted Tomato retail price.
4. Test the API locally.
5. Create a Dockerfile and requirements file.
6. Build and run the API inside a Docker container.

## Model used
The final model from Experiment 4 is a **Tuned Random Forest Regressor** for **Tomato retail-price prediction**.

The model uses these eight forecasting-safe features:
- `Tomato_Lag1`
- `Tomato_Lag2`
- `Tomato_PastRollingMean3`
- `Month`
- `Quarter`
- `Week`
- `Month_Sin`
- `Month_Cos`

In [1]:
%pip install -q fastapi==0.128.2 uvicorn[standard]==0.40.0 pydantic==2.13.4 httpx==0.28.1
print("FastAPI dependencies are ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires fastapi<1,>=0.133, but you have fastapi 0.128.2 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.50.0 which is incompatible.
python-fasthtml 0.14.12 requires starlette>=1.0.1, but you have starlette 0.50.0 which is incompatible.
gradio 6.26.0 requires starlette<2.0,>=1.0.1, but

In [3]:
from pathlib import Path
import json
import shutil
import joblib
import pandas as pd

BASE_DIR = Path.cwd()
MODEL_SOURCE = Path("/content/best_model.pkl")

# In Colab, place best_model.pkl in /content before running the next cells.
if not (BASE_DIR / "best_model.pkl").exists() and MODEL_SOURCE.exists():
    shutil.copy2(MODEL_SOURCE, BASE_DIR / "best_model.pkl")

print("Working directory:", BASE_DIR)
print("Model exists:", (BASE_DIR / "best_model.pkl").exists())

model = joblib.load(BASE_DIR / "best_model.pkl")

FEATURES = [
    "Tomato_Lag1",
    "Tomato_Lag2",
    "Tomato_PastRollingMean3",
    "Month",
    "Quarter",
    "Week",
    "Month_Sin",
    "Month_Cos",
]

print("Model type:", type(model).__name__)
print("Model feature names:", list(getattr(model, "feature_names_in_", FEATURES)))

Working directory: /content
Model exists: True
Model type: RandomForestRegressor
Model feature names: ['Tomato_Lag1', 'Tomato_Lag2', 'Tomato_PastRollingMean3', 'Month', 'Quarter', 'Week', 'Month_Sin', 'Month_Cos']


## 1. FastAPI Application

The API exposes:
- `GET /` – basic information
- `GET /health` – model health check
- `POST /predict` – prediction endpoint

In [4]:
main_py = 'from pathlib import Path\n\nimport joblib\nimport pandas as pd\nfrom fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel, Field\n\nAPP_DIR = Path(__file__).resolve().parent\nMODEL_PATH = APP_DIR / "best_model.pkl"\n\nFEATURES = [\n    "Tomato_Lag1",\n    "Tomato_Lag2",\n    "Tomato_PastRollingMean3",\n    "Month",\n    "Quarter",\n    "Week",\n    "Month_Sin",\n    "Month_Cos",\n]\n\napp = FastAPI(\n    title="Essential Food Item Inflation Forecaster API",\n    description="FastAPI service for Tomato retail-price prediction using the tuned Random Forest model from Experiment 4.",\n    version="1.0.0",\n)\n\ntry:\n    model = joblib.load(MODEL_PATH)\nexcept Exception as exc:\n    model = None\n    MODEL_LOAD_ERROR = str(exc)\n\n\nclass PredictionRequest(BaseModel):\n    Tomato_Lag1: float = Field(..., description="Tomato price at the previous observation")\n    Tomato_Lag2: float = Field(..., description="Tomato price two observations earlier")\n    Tomato_PastRollingMean3: float = Field(..., description="Mean of the previous three Tomato prices")\n    Month: int = Field(..., ge=1, le=12)\n    Quarter: int = Field(..., ge=1, le=4)\n    Week: int = Field(..., ge=1, le=53)\n    Month_Sin: float\n    Month_Cos: float\n\n\n@app.get("/")\ndef root():\n    return {\n        "message": "Essential Food Item Inflation Forecaster API",\n        "target": "Tomato retail price",\n        "problem_type": "Regression",\n        "endpoint": "/predict",\n    }\n\n\n@app.get("/health")\ndef health():\n    if model is None:\n        return {"status": "error", "model_loaded": False, "error": MODEL_LOAD_ERROR}\n    return {"status": "ok", "model_loaded": True}\n\n\n@app.post("/predict")\ndef predict(request: PredictionRequest):\n    if model is None:\n        raise HTTPException(status_code=500, detail=f"Model could not be loaded: {MODEL_LOAD_ERROR}")\n\n    input_df = pd.DataFrame(\n        [[\n            request.Tomato_Lag1,\n            request.Tomato_Lag2,\n            request.Tomato_PastRollingMean3,\n            request.Month,\n            request.Quarter,\n            request.Week,\n            request.Month_Sin,\n            request.Month_Cos,\n        ]],\n        columns=FEATURES,\n    )\n\n    try:\n        prediction = float(model.predict(input_df)[0])\n    except Exception as exc:\n        raise HTTPException(status_code=400, detail=f"Prediction failed: {exc}")\n\n    return {\n        "predicted_tomato_price": round(prediction, 4),\n        "unit": "INR/kg",\n        "model": "Tuned Random Forest Regressor",\n        "features_used": FEATURES,\n    }\n'
(BASE_DIR / "main.py").write_text(main_py, encoding="utf-8")
print("Created:", BASE_DIR / "main.py")

Created: /content/main.py


In [5]:
print((BASE_DIR / "main.py").read_text(encoding="utf-8"))

from pathlib import Path

import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

APP_DIR = Path(__file__).resolve().parent
MODEL_PATH = APP_DIR / "best_model.pkl"

FEATURES = [
    "Tomato_Lag1",
    "Tomato_Lag2",
    "Tomato_PastRollingMean3",
    "Month",
    "Quarter",
    "Week",
    "Month_Sin",
    "Month_Cos",
]

app = FastAPI(
    title="Essential Food Item Inflation Forecaster API",
    description="FastAPI service for Tomato retail-price prediction using the tuned Random Forest model from Experiment 4.",
    version="1.0.0",
)

try:
    model = joblib.load(MODEL_PATH)
except Exception as exc:
    model = None
    MODEL_LOAD_ERROR = str(exc)


class PredictionRequest(BaseModel):
    Tomato_Lag1: float = Field(..., description="Tomato price at the previous observation")
    Tomato_Lag2: float = Field(..., description="Tomato price two observations earlier")
    Tomato_PastRollingMean3: float = Field(..., description=

## 2. Sample JSON Request

In [6]:
sample_input = {'Tomato_Lag1': 47.61, 'Tomato_Lag2': 46.47, 'Tomato_PastRollingMean3': 47.43, 'Month': 7, 'Quarter': 3, 'Week': 29, 'Month_Sin': -0.5, 'Month_Cos': -0.8660254038}
(BASE_DIR / "sample_input.json").write_text(json.dumps(sample_input, indent=2), encoding="utf-8")
print(json.dumps(sample_input, indent=2))

{
  "Tomato_Lag1": 47.61,
  "Tomato_Lag2": 46.47,
  "Tomato_PastRollingMean3": 47.43,
  "Month": 7,
  "Quarter": 3,
  "Week": 29,
  "Month_Sin": -0.5,
  "Month_Cos": -0.8660254038
}


## 3. Local API Test

FastAPI's `TestClient` is used here so the endpoint can be tested inside the notebook without needing Docker.

In [7]:
from fastapi.testclient import TestClient
import importlib.util
import sys

spec = importlib.util.spec_from_file_location("main", BASE_DIR / "main.py")
main_module = importlib.util.module_from_spec(spec)
sys.modules["main"] = main_module
spec.loader.exec_module(main_module)

client = TestClient(main_module.app)

health_response = client.get("/health")
prediction_response = client.post("/predict", json=sample_input)

print("Health status code:", health_response.status_code)
print("Health response:", health_response.json())

print("\nPrediction status code:", prediction_response.status_code)
print("Prediction response:", prediction_response.json())

Health status code: 200
Health response: {'status': 'ok', 'model_loaded': True}

Prediction status code: 200
Prediction response: {'predicted_tomato_price': 50.0924, 'unit': 'INR/kg', 'model': 'Tuned Random Forest Regressor', 'features_used': ['Tomato_Lag1', 'Tomato_Lag2', 'Tomato_PastRollingMean3', 'Month', 'Quarter', 'Week', 'Month_Sin', 'Month_Cos']}


In [8]:
check_df = pd.DataFrame([sample_input], columns=FEATURES)
direct_prediction = float(model.predict(check_df)[0])

print("Direct model prediction:", round(direct_prediction, 4))
print("API prediction:", prediction_response.json()["predicted_tomato_price"])

Direct model prediction: 50.0924
API prediction: 50.0924


## 4. Run the API Locally

From a terminal in the experiment folder:

```bash
uvicorn main:app --host 127.0.0.1 --port 8000
```

Open:

```text
http://127.0.0.1:8000/docs
```

The Swagger UI allows interactive testing of `/predict`.

In [9]:
test_api_py = 'import requests\n\nurl = "http://127.0.0.1:8000/predict"\n\npayload = {\n    "Tomato_Lag1": 47.61,\n    "Tomato_Lag2": 46.47,\n    "Tomato_PastRollingMean3": 47.43,\n    "Month": 7,\n    "Quarter": 3,\n    "Week": 29,\n    "Month_Sin": -0.5,\n    "Month_Cos": -0.8660254038\n}\n\nresponse = requests.post(url, json=payload, timeout=10)\n\nprint("Status code:", response.status_code)\nprint("Response:")\nprint(response.json())\nresponse.raise_for_status()\n'
(BASE_DIR / "test_api.py").write_text(test_api_py, encoding="utf-8")
print("Created:", BASE_DIR / "test_api.py")

Created: /content/test_api.py


## 5. Dockerfile

```dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY best_model.pkl .
COPY main.py .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

In [10]:
requirements_txt = 'fastapi==0.128.2\nuvicorn[standard]==0.40.0\npydantic==2.13.4\npandas==2.3.3\nnumpy==2.2.6\nscikit-learn==1.6.1\njoblib==1.5.2\nrequests==2.32.5\nhttpx==0.28.1\n'
dockerfile = 'FROM python:3.10-slim\n\nWORKDIR /app\n\nCOPY requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\n\nCOPY best_model.pkl .\nCOPY main.py .\n\nEXPOSE 8000\n\nCMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]\n'

(BASE_DIR / "requirements.txt").write_text(requirements_txt, encoding="utf-8")
(BASE_DIR / "Dockerfile").write_text(dockerfile, encoding="utf-8")

print("Created requirements.txt and Dockerfile.")

Created requirements.txt and Dockerfile.


## 6. Build and Test Docker Container

Run these commands on a system with Docker installed:

### Build image
```bash
docker build -t food-price-api .
```

### Run container
```bash
docker run -d --name food-price-api-container -p 8000:8000 food-price-api
```

### Test
```bash
python test_api.py
```

### Stop and remove
```bash
docker stop food-price-api-container
docker rm food-price-api-container
```

## 7. Expected API Response

A successful request returns JSON similar to:

```json
{
  "predicted_tomato_price": 50.0924,
  "unit": "INR/kg",
  "model": "Tuned Random Forest Regressor"
}
```

The exact value depends on the supplied `best_model.pkl` artifact.

## Deliverables

- `main.py` – FastAPI API code
- `best_model.pkl` – trained model
- `requirements.txt` – dependencies
- `Dockerfile` – container configuration
- `sample_input.json` – sample POST input
- `test_api.py` – local API test
- Docker build/run commands
- Local API test evidence

## Conclusion

The final machine learning model was packaged as a FastAPI prediction service. The `/predict` endpoint accepts the same eight input features used during Experiment 4 and returns the predicted Tomato retail price. The API was tested locally, and a Dockerfile was prepared so the service and model can be deployed as a portable container. This creates a deployment-ready interface for future integration with the project's dashboard or other applications.